# 02_demand — спрос по поиску (роль B: вопросы 2 и 3)

**Вопрос 2.** Можно ли верить поиску: опережает ли он экспорт K-beauty из Кореи в страны СНГ и на сколько месяцев?
**Вопрос 3.** Какие категории и ингредиенты растут и чем отличаются страны?

Данные: `data/clean/panel_search_export.csv`, `search_clean.csv`, `keywords.csv` (собраны `build_B_clean.py`), таможня — `customs_clean.csv` (Гузель), снимок Google Trends по странам — `data/raw/04_trends_geo_12m_expanded.csv`, Naver DataLab (Корея).
Период поиска 2020-01 … 2026-10. Общие функции — `utils.py`.

In [ ]:
import sys, warnings, io, contextlib
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'utils.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from utils import (clean_path, raw_path, time_split, yoy_log, zero_share, load_calendar,
                   PALETTE, ISO2_TO_RU, RANDOM_STATE)

FIG = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
mpl.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 160, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
GEOS = ['RU', 'KZ', 'BY', 'KG', 'UZ']
RU_NAME = {g: ISO2_TO_RU[g] for g in GEOS}
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

## Вопрос 2. Поиск как индикатор спроса: опережает ли он экспорт?

**Y** — рост экспорта в тоннах год к году в месяц t (`tons_yoy` = log(тонны_t + 1) − log(тонны_{t−12} + 1)).
**X** — рост поиска год к году в месяцы t−1 … t−6. Поиск: Россия — Вордстат (абсолютные числа), остальные — Google Trends (индекс). Шкалы разные, поэтому везде сравниваем только **рост г/г**, а не уровни.

### 2.1 Data preprocessing

In [ ]:
panel = pd.read_csv(clean_path('panel_search_export.csv'), dtype={'month': str})
search = pd.read_csv(clean_path('search_clean.csv'), dtype={'month': str})
customs = pd.read_csv(clean_path('customs_clean.csv'), dtype={'month': str}, encoding='utf-8-sig')
cal = load_calendar()
print('panel:', panel.shape, panel.month.min(), '…', panel.month.max())
print('таможня из customs_clean.csv:', customs.shape)
panel.head(3)

Доля нулей в рядах поиска, которые идут в вопрос 2 (ряд с нулями > 50% в модель не берём):

In [ ]:
q2_series = pd.concat([
    search[(search.export_id == 'T01')].assign(kind='общий запрос'),
    search[(search.source == 'wordstat') & (search.term == 'корейская косметика')].assign(kind='общий запрос'),
    search[search.export_id == 'without_medicube'].assign(kind='бренды без Medicube'),
    search[search.export_id == 'with_medicube'].assign(kind='бренды с Medicube'),
])
zs = q2_series.groupby(['geo', 'kind'])['zero_share'].first().unstack().reindex(GEOS)
zs['бренды Вордстат (RU)'] = np.nan
zs.loc['RU', ['бренды без Medicube', 'бренды с Medicube']] = 0.0   # сумма брендов Вордстата: абсолютные числа без нулей
(zs.drop(columns='бренды Вордстат (RU)') * 100).round(0).astype('Int64').rename(index=RU_NAME).style.format('{}%')

### 2.2 EDA

Сначала — предупреждение: на **уровнях** два растущих ряда коррелируют «сами по себе» (общий тренд). Поэтому вывод делаем только по росту г/г.

In [ ]:
lvl = []
for g in GEOS:
    d = panel[panel.iso2 == g]
    lvl.append({'страна': RU_NAME[g],
                'r уровни (тонны ~ поиск)': d['tons'].corr(d['search_general']),
                'r рост г/г, тот же месяц': d['tons_yoy'].corr(d['search_general_yoy'])})
pd.DataFrame(lvl).set_index('страна').round(2)

In [ ]:
def lag_corr(df, col, method='pearson', start='2021-01'):
    '''r(поиск_{t-k}, тонны_t) на росте г/г, k = 0…6, по каждой стране.'''
    out = {}
    for g in GEOS:
        d = df[(df.iso2 == g) & (df.month >= start)]
        row = {}
        for k in range(7):
            x = d[f'{col}_yoy'] if k == 0 else d[f'{col}_yoy_lag{k}']
            m = x.notna() & d['tons_yoy'].notna()
            r, p = (stats.pearsonr if method == 'pearson' else stats.spearmanr)(x[m], d['tons_yoy'][m])
            row[k] = (r, p, m.sum())
        out[g] = row
    r = pd.DataFrame({g: {k: v[0] for k, v in row.items()} for g, row in out.items()}).T
    p = pd.DataFrame({g: {k: v[1] for k, v in row.items()} for g, row in out.items()}).T
    return r, p

# бренды с Medicube: лаги считаем здесь же (в панели лаги есть для general и brands)
for k in range(1, 7):
    panel[f'search_brands_medicube_yoy_lag{k}'] = np.nan
panel['search_brands_medicube_yoy'] = panel.groupby('iso2')['search_brands_medicube'].transform(yoy_log)
full = pd.read_csv(clean_path('panel_search_export.csv'), dtype={'month': str})  # 2020 нужен для лагов роста
for g in GEOS:
    m = panel.iso2 == g
    s = panel.loc[m, 'search_brands_medicube_yoy']
    for k in range(1, 7):
        panel.loc[m, f'search_brands_medicube_yoy_lag{k}'] = s.shift(k).values

r_gen, p_gen = lag_corr(panel, 'search_general')
r_gen_s, _ = lag_corr(panel, 'search_general', 'spearman')
r_br, p_br = lag_corr(panel, 'search_brands')
r_brm, p_brm = lag_corr(panel, 'search_brands_medicube')

def show(r, p):
    t = r.round(2).astype(str) + np.where(p < 0.05, '*', '')
    return t.rename(index=RU_NAME, columns=lambda k: f'лаг {k}')
print('Пирсон, общий запрос (* — p < 0.05):'); display(show(r_gen, p_gen))
print('Спирмен, общий запрос:'); display(r_gen_s.round(2).rename(index=RU_NAME, columns=lambda k: f'лаг {k}'))

In [ ]:
best = pd.DataFrame({
    'общий запрос': r_gen.max(axis=1),
    'бренды без Medicube': r_br.max(axis=1),
    'бренды с Medicube': r_brm.max(axis=1),
}).rename(index=RU_NAME).round(2)
print('Лучшая (максимальная) корреляция по лагам 0–6 — какой поиск лучше следит за экспортом:')
best

### 2.3 Visualisation

In [ ]:
# Главный для слайда 6: heatmap «страна × лаг»
fig, ax = plt.subplots(figsize=(7.2, 3.4))
M = r_gen.loc[GEOS, 1:6]
im = ax.imshow(M.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_xticks(range(6), [f'{k} мес' for k in range(1, 7)])
ax.set_yticks(range(len(GEOS)), [RU_NAME[g] for g in GEOS])
for i, g in enumerate(GEOS):
    for j, k in enumerate(range(1, 7)):
        sig = p_gen.loc[g, k] < 0.05
        ax.text(j, i, f'{M.iloc[i, j]:+.2f}' + ('*' if sig else ''), ha='center', va='center',
                fontsize=9, fontweight='bold' if sig else 'normal')
ax.set_xlabel('на сколько месяцев поиск раньше отгрузок')
ax.set_title('Связь роста поиска и роста экспорта (тонны), г/г, 2021–2026\n* — p < 0,05', loc='left', fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.8, label='r Пирсона')
fig.tight_layout(); fig.savefig(FIG / 'q2_heatmap_lag.png'); plt.show()

In [ ]:
# Поиск и экспорт на двух осях: Россия и Казахстан (скользящее среднее за 3 прошлых месяца — только для читаемости)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, g in zip(axes, ['RU', 'KZ']):
    d = panel[panel.iso2 == g].copy()
    x = pd.to_datetime(d.month)
    ax.plot(x, d.tons.rolling(3).mean(), color=PALETTE[g], lw=2, label='экспорт, т')
    ax.set_ylabel('экспорт, тонн в месяц', color=PALETTE[g])
    ax2 = ax.twinx(); ax2.spines['right'].set_visible(True)
    ax2.plot(x, d.search_general.rolling(3).mean(), color='#444', lw=1.3, ls='--', label='поиск')
    ax2.set_ylabel('запросов в месяц (Вордстат)' if g == 'RU' else 'индекс Google Trends', color='#444')
    ax.set_title(RU_NAME[g] + ': «корейская косметика» и отгрузки HS 3304', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG / 'q2_dual_axis_ru_kz.png'); plt.show()

In [ ]:
# Scatter «поиск t−k → экспорт t» для страны и лага с самой сильной связью
g_best, k_best = r_gen.loc[:, 1:6].stack().idxmax()
d = panel[(panel.iso2 == g_best) & (panel.month >= '2021-01')].dropna(subset=['tons_yoy', f'search_general_yoy_lag{k_best}'])
x, y = d[f'search_general_yoy_lag{k_best}'], d['tons_yoy']
b = np.polyfit(x, y, 1)
fig, ax = plt.subplots(figsize=(5.2, 4))
ax.scatter(x, y, color=PALETTE[g_best], alpha=0.7)
xx = np.linspace(x.min(), x.max(), 50); ax.plot(xx, np.polyval(b, xx), color='k', lw=1)
ax.axhline(0, color='#bbb', lw=0.6); ax.axvline(0, color='#bbb', lw=0.6)
ax.set_xlabel(f'рост поиска г/г, {k_best} мес назад (log)'); ax.set_ylabel('рост экспорта в тоннах г/г (log)')
ax.set_title(f'{RU_NAME[g_best]}: r = {r_gen.loc[g_best, k_best]:.2f}, лаг {k_best} мес', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG / 'q2_scatter_best.png'); plt.show()

### 2.4 ML models

**Главная модель:** Y = рост тонн г/г в месяц t ~ рост поиска t−1…t−6 + месяц (dummy) + страна (one-hot). Порядок: база → Linear → Ridge → Random Forest. База — та же модель **без поиска** и наивная «как год назад» (рост = 0).
Train 2021-01 … 2024-12, test 2025-01 … 2026-10, разбиение только по времени (`time_split` из utils). Масштабирование — внутри Pipeline (учится только на train).

In [ ]:
LAGS = [f'search_general_yoy_lag{k}' for k in range(1, 7)]
data = panel[(panel.month >= '2021-01')].dropna(subset=['tons_yoy'] + LAGS).copy()
data['mon'] = data.month.str[5:7].astype(int)
X_base = pd.get_dummies(data[['mon', 'iso2']].astype(str), drop_first=True).astype(float)
X_full = pd.concat([X_base, data[LAGS]], axis=1)
data = pd.concat([data[['month', 'iso2', 'tons_yoy']], X_full], axis=1)
train, test = time_split(data, '2024-12', '2025-01', gap=0)
base_cols, full_cols = list(X_base.columns), list(X_full.columns)
print('train', train.shape, train.month.min(), '…', train.month.max(), '| test', test.shape, test.month.min(), '…', test.month.max())

def metrics(name, cols, model):
    model.fit(train[cols], train.tons_yoy)
    row = {'модель': name}
    for part, d in [('train', train), ('test', test)]:
        pred = model.predict(d[cols])
        row[f'MAE {part}'] = mean_absolute_error(d.tons_yoy, pred)
        row[f'RMSE {part}'] = mean_squared_error(d.tons_yoy, pred) ** 0.5
        row[f'R² {part}'] = r2_score(d.tons_yoy, pred)
    return row

class Naive:            # «как год назад»: тонны те же, рост г/г = 0
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X))

tscv = TimeSeriesSplit(n_splits=4)
ridge = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': [0.1, 1, 10, 100]}, cv=tscv,
                     scoring='neg_mean_absolute_error')
rf_grid = {'max_depth': [2, 3, 5], 'min_samples_leaf': [5, 10, 20]}
rows = [metrics('Наивная «как год назад»', base_cols, Naive()),
        metrics('Linear без поиска (месяц + страна)', base_cols, make_pipeline(StandardScaler(), LinearRegression())),
        metrics('Linear с поиском', full_cols, make_pipeline(StandardScaler(), LinearRegression())),
        metrics('Ridge без поиска', base_cols, GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {'ridge__alpha': [0.1, 1, 10, 100]}, cv=tscv, scoring='neg_mean_absolute_error')),
        metrics('Ridge с поиском', full_cols, ridge),
        metrics('RF без поиска', base_cols, GridSearchCV(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE), rf_grid, cv=tscv, scoring='neg_mean_absolute_error')),
        metrics('RF с поиском', full_cols, GridSearchCV(RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE), rf_grid, cv=tscv, scoring='neg_mean_absolute_error'))]
m2 = pd.DataFrame(rows).set_index('модель').round(3)
m2

**Лучший лаг по странам** выбираем на train (а не на test): максимальная корреляция роста поиска t−k и роста тонн на 2021–2024. Затем на test сравниваем MAE простой регрессии `tons_yoy ~ поиск_{t−k}` с моделью без поиска (средний рост на train). Это и есть `lead_lag.csv` для Гузель (календарь) и итоговой карточки.

In [ ]:
rows = []
for g in GEOS:
    d = panel[(panel.iso2 == g) & (panel.month >= '2021-01')]
    tr, te = time_split(d, '2024-12', '2025-01')
    rs = {k: tr['tons_yoy'].corr(tr[f'search_general_yoy_lag{k}']) for k in range(1, 7)}
    k = max(rs, key=rs.get)
    col = f'search_general_yoy_lag{k}'
    trk, tek = tr.dropna(subset=[col, 'tons_yoy']), te.dropna(subset=[col, 'tons_yoy'])
    lr = LinearRegression().fit(trk[[col]], trk.tons_yoy)
    mae_w = mean_absolute_error(tek.tons_yoy, lr.predict(tek[[col]]))
    mae_wo = mean_absolute_error(tek.tons_yoy, np.full(len(tek), trk.tons_yoy.mean()))
    r_test = tek['tons_yoy'].corr(tek[col])
    rows.append({'iso2': g, 'best_lag': k, 'r': round(rs[k], 3), 'r_test': round(r_test, 3),
                 'mae_with_search': round(mae_w, 4), 'mae_without_search': round(mae_wo, 4),
                 'mae_change_pct': round((mae_w / mae_wo - 1) * 100, 1)})
lead_lag = pd.DataFrame(rows)
lead_lag

**Проверка 1: тест Грейнджера.** Сначала ADF-тест: стационарны ли ряды роста г/г (p < 0,05 → да). Затем `grangercausalitytests` (statsmodels): помогает ли прошлый поиск предсказать экспорт сверх прошлого экспорта, лаги 1–6. Минимальное p по лагам и лаг, где оно достигнуто.
**Проверка 2:** то же без 2020–2021 (только 2022-01 … 2026-10).

In [ ]:
def granger_table(start):
    rows = []
    for g in GEOS:
        d = panel[(panel.iso2 == g) & (panel.month >= start)][['tons_yoy', 'search_general_yoy']].dropna()
        adf_y, adf_x = adfuller(d.tons_yoy)[1], adfuller(d.search_general_yoy)[1]
        with contextlib.redirect_stdout(io.StringIO()):
            res = grangercausalitytests(d[['tons_yoy', 'search_general_yoy']], maxlag=6)
        ps = {k: res[k][0]['ssr_ftest'][1] for k in res}
        k = min(ps, key=ps.get)
        rows.append({'страна': RU_NAME[g], 'месяцев': len(d), 'ADF p экспорт': adf_y, 'ADF p поиск': adf_x,
                     'Грейнджер: лучший лаг': k, 'p': ps[k]})
    return pd.DataFrame(rows).set_index('страна').round(3)
g_all = granger_table('2021-01'); g_22 = granger_table('2022-01')
print('2021-01 … 2026-10'); display(g_all)
print('Без 2020–2021: 2022-01 … 2026-10'); display(g_22)

In [ ]:
lead_lag['granger_p'] = lead_lag.iso2.map(dict(zip(GEOS, g_all['p'].values)))
lead_lag['granger_p_2022'] = lead_lag.iso2.map(dict(zip(GEOS, g_22['p'].values)))
lead_lag['trust'] = np.where((lead_lag.r > 0) & (lead_lag.granger_p < 0.05) & (lead_lag.mae_change_pct < 0), 'средняя',
                             'низкая')
lead_lag['note'] = np.where(lead_lag.iso2.isin(['BY', 'KG', 'UZ']), 'мало данных (Trends, много нулей в брендах)', '')
lead_lag.to_csv(clean_path('lead_lag.csv'), index=False)
lead_lag

In [ ]:
ll = lead_lag.set_index('iso2')
def line(g):
    r = ll.loc[g]
    return (f"{RU_NAME[g]}: лучший лаг {int(r.best_lag)} мес, r на train {r.r:+.2f}, на test {r.r_test:+.2f}; "
            f"MAE с поиском {r.mae_change_pct:+.0f}% к модели без поиска; Грейнджер p = {r.granger_p:.3f}")
print('\n'.join(line(g) for g in GEOS))

### Вывод по вопросу 2 (для стратегии)

Числа — на данных 26.11.2026 (таблицы выше и `lead_lag.csv`).

- **Казахстан — поиск опережает отгрузки на 2–5 месяцев.** Рост запроса «корейская косметика» связан с ростом тонн через 2 мес (r = +0,38) и 4 мес (r = +0,31), оба значимы; тест Грейнджера значим (p = 0,004, без 2020–2021 p = 0,001). Лучший лаг, выбранный на train, — 2 мес; на test он снижает ошибку прогноза на 14% по сравнению с моделью без поиска. → Выводам 3–5 для Казахстана доверяем со **средней** уверенностью; сдвиг k = 2 мес идёт в календарь (вопрос 5).
- **Россия — связи нет.** Корреляции на всех лагах от −0,17 до 0, Грейнджер незначим (p = 0,42), поиск не улучшает прогноз (MAE +0,4%). Отгрузки в Россию задают логистика, санкции и маркировка, а не текущий интерес. → Поиск в России показывает, **что** ищут (вопрос 3), но не **когда** придут отгрузки; уверенность **низкая**.
- **Беларусь, Кыргызстан, Узбекистан — связи нет** (на test поиск ухудшает ошибку на 9–26%), мало данных.
- **Общая модель** по 5 странам (Linear / Ridge / RF, лаги 1–6 + месяц + страна): с поиском на test не лучше, чем без поиска (MAE 0,40–0,42 против 0,39–0,40), R² на test около 0 у всех → поиск сам по себе экспорт не прогнозирует.
- **Находка:** общий запрос «корейская косметика» падает (Россия −15% в год, доля в запросах 15% → 4,5%), а запросы брендов растут (доля 6% → 11%). Люди ищут уже конкретные бренды, поэтому общий запрос плохо отражает спрос; бренды как индикатор тоже не опережают тонны.

## Вопрос 3. Какие категории и ингредиенты растут и чем отличаются страны?

Динамика: Россия — Вордстат (25 ингредиентов, 13 категорий, 15 брендов, абсолютные числа); Казахстан, Беларусь, Кыргызстан, Узбекистан — Google Trends (5 ингредиентов, индекс своей страны). Сравнение стран между собой — снимок за 12 месяцев (файл 04) + последние 12 месяцев Вордстата для России.

### 3.1 Data preprocessing

In [ ]:
ws = search[search.source == 'wordstat'].copy()          # Россия, абсолютные числа
ing = search[search.export_id == 'ing'].copy()          # 4 страны, 5 ингредиентов, свой индекс в каждой стране
kw = pd.read_csv(clean_path('keywords.csv'))

# индекс «2021 = 100» для Вордстата (чтобы сравнивать направление с Trends)
base21 = ws[ws.month.str[:4] == '2021'].groupby('term')['value'].mean()
ws['idx2021'] = ws['value'] / ws['term'].map(base21) * 100

def growth(df, val='value'):
    y = df.month.str[:4]; m = df.month.str[5:7]
    s21 = df[y == '2021'].groupby(key)[val].sum(); s25 = df[y == '2025'].groupby(key)[val].sum()
    jo25 = df[(y == '2025') & (m <= '10')].groupby(key)[val].sum(); jo26 = df[(y == '2026')].groupby(key)[val].sum()
    return pd.DataFrame({'growth_2025_vs_2021': s25 / s21 - 1, 'growth_2026_vs_2025': jo26 / jo25 - 1})

key = ['source', 'geo', 'group', 'term']
both = pd.concat([ws, ing])
gt = growth(both)
gt['zero_share'] = both.groupby(key)['zero_share'].first()
gt['enough_data'] = gt.zero_share <= 0.5
gt['volume_2025'] = both[both.month.str[:4] == '2025'].groupby(key)['value'].sum()
gt.replace([np.inf, -np.inf], np.nan, inplace=True)
print(gt.shape); gt.head()

**Тренд-регрессия на каждый ряд (главная модель вопроса 3):** Y = log(индекс + 1), X = номер месяца + месяцы-dummy (чтобы сезонность не путалась с ростом). Наклон → % роста в год = exp(12·b) − 1; p-value с ошибками Newey–West (HAC, 3 лага), потому что соседние месяцы зависимы. Две оценки: весь период 2020–2026 и последние 24 месяца (куда ряд идёт сейчас).

In [ ]:
def trend(d):
    d = d.sort_values('month')
    y = np.log1p(d['value'].values.astype(float))
    t = np.arange(len(d))
    dm = pd.get_dummies(d.month.str[5:7], drop_first=True).astype(float).values
    X = sm.add_constant(np.column_stack([t, dm]))
    fit = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
    return np.exp(12 * fit.params[1]) - 1, fit.pvalues[1]

rows = []
for k, d in both.groupby(key):
    if d.zero_share.iloc[0] > 0.5:
        continue
    full_g, full_p = trend(d)
    last_g, last_p = trend(d[d.month >= '2024-11'])
    rows.append(dict(zip(key, k), trend_year_pct=full_g * 100, trend_p=full_p, trend24_year_pct=last_g * 100, trend24_p=last_p))
tr = pd.DataFrame(rows).set_index(key)
growth_table = gt.join(tr).reset_index()
growth_table['term_en'] = growth_table.term.map(kw.drop_duplicates('term').set_index('term')['term_en'])
growth_table['k_specific'] = growth_table.term.map(kw.drop_duplicates('term').set_index('term')['k_specific']).fillna(0).astype(int)
growth_table.round(4).to_csv(clean_path('growth_table.csv'), index=False)
print('growth_table.csv:', growth_table.shape)

### 3.2 EDA — что растёт и что падает

In [ ]:
ru = growth_table[(growth_table.source == 'wordstat') & growth_table.trend_p.notna()].copy()
cols = ['group', 'term', 'growth_2026_vs_2025', 'growth_2025_vs_2021', 'trend_year_pct', 'trend_p', 'trend24_year_pct', 'trend24_p']
fmt = {'growth_2026_vs_2025': '{:+.0%}', 'growth_2025_vs_2021': '{:+.0%}', 'trend_year_pct': '{:+.0f}%',
       'trend_p': '{:.3f}', 'trend24_year_pct': '{:+.0f}%', 'trend24_p': '{:.3f}'}
print('Россия — топ-10 растущих (по тренду последних 24 мес):')
display(ru.sort_values('trend24_year_pct', ascending=False)[cols].head(10).style.format(fmt).hide(axis='index'))
print('Россия — топ-10 падающих:')
display(ru.sort_values('trend24_year_pct')[cols].head(10).style.format(fmt).hide(axis='index'))

In [ ]:
other = growth_table[(growth_table.source == 'trends')].copy()
print('Казахстан, Беларусь, Кыргызстан, Узбекистан — 5 ингредиентов (ряды с нулями > 50% помечены):')
o = other[['geo', 'term', 'zero_share', 'growth_2026_vs_2025', 'trend_year_pct', 'trend_p', 'trend24_year_pct', 'trend24_p']].copy()
o['geo'] = o.geo.map(RU_NAME)
o.sort_values(['geo', 'trend24_year_pct'], ascending=[True, False]).style.format(
    {'zero_share': '{:.0%}', 'growth_2026_vs_2025': '{:+.0%}', 'trend_year_pct': '{:+.0f}%', 'trend_p': '{:.3f}',
     'trend24_year_pct': '{:+.0f}%', 'trend24_p': '{:.3f}'}, na_rep='—').hide(axis='index')

In [ ]:
# Доля каждой группы в общем спросе России по годам (Вордстат, сумма запросов)
sh = ws[ws.month <= '2026-10'].assign(year=ws.month.str[:4]).groupby(['year', 'group'])['value'].sum().unstack()
sh = sh.div(sh.sum(axis=1), axis=0) * 100
sh.columns = sh.columns.map({'brand': 'бренды', 'category': 'категории', 'general': '«корейская косметика»', 'ingredient': 'ингредиенты'})
print('Россия: доля групп в запросах, % (2026 — янв–окт)'); sh.round(1)

**5 общих ингредиентов: Россия против 4 стран.** Сравниваем только направление и скорость (рост г/г в логарифмах), не уровни. Крускал–Уоллис: различается ли распределение месячного роста г/г (2022–2026) между странами для одного ингредиента. Ряды с нулями > 50% не берём.

In [ ]:
COMMON = ['pdrn', 'ниацинамид', 'центелла', 'ретинол', 'азелаиновая кислота']
kw_rows = []
for term in COMMON:
    groups, names = [], []
    for g in GEOS:
        d = (ws[ws.term == term] if g == 'RU' else ing[(ing.geo == g) & (ing.term == term)]).sort_values('month')
        if len(d) == 0 or d.zero_share.iloc[0] > 0.5:
            continue
        yy = yoy_log(d['value'].values)
        yy = pd.Series(yy.values, index=d.month.values)
        groups.append(yy[yy.index >= '2022-01'].dropna().values); names.append(g)
    if len(groups) >= 2:
        H, p = stats.kruskal(*groups)
        med = {RU_NAME[n]: np.median(gg) for n, gg in zip(names, groups)}
        kw_rows.append({'ингредиент': term, 'страны': ', '.join(RU_NAME[n] for n in names), 'H': H, 'p': p,
                        **{f'медиана роста г/г, {k}': v for k, v in med.items()}})
kruskal = pd.DataFrame(kw_rows).set_index('ингредиент')
kruskal.round(3)

### 3.3 Visualisation и профили стран

In [ ]:
# Профиль стран из снимка 04 (12 мес): внутри каждой выгрузки 5 слов делят 100%.
# Чтобы склеить выгрузки, делим на «якорь» (anua — бренды, ниацинамид — ингредиенты, сыворотка для лица — категории),
# якорь = 100 (так страны сравнимы, даже если у страны нет части слов). Россия — те же 5 слов из Вордстата за последние 12 мес (2025-11 … 2026-10).
g04 = pd.read_csv(raw_path('04_trends_geo_12m_expanded.csv'), encoding='utf-8-sig')
g04['share'] = np.where(g04.hasData, g04.formattedValue.str.replace('<1%', '0.5').str.replace(' %', '').astype(float), 0.0)
g04 = g04.groupby(['query_id', 'geoCode', 'term'], as_index=False)['share'].first()
QGROUP = {'Q14': 'brand', 'Q15': 'brand', 'Q16': 'brand', 'Q17': 'brand', 'Q18': 'ingredient', 'Q19': 'ingredient',
          'Q20': 'category', 'Q21': 'category', 'Q22': 'category', 'Q23': 'category', 'Q24': 'category'}
ANCHOR = {'brand': 'anua', 'ingredient': 'ниацинамид', 'category': 'сыворотка для лица'}
g04['group'] = g04.query_id.map(QGROUP)
g04['term'] = [t if t != 'санскрин' else 'солнцезащитный крем' for t in g04.term]

# Россия: та же форма из Вордстата
last12 = ws[ws.month >= '2025-11'].groupby('term')['value'].sum()
ru_rows = []
for q, d in g04[g04.geoCode == 'KZ'].groupby('query_id'):
    terms = list(d.term)
    if all(t in last12.index for t in terms):
        tot = last12[terms].sum()
        ru_rows += [{'query_id': q, 'geoCode': 'RU', 'term': t, 'share': last12[t] / tot * 100, 'group': QGROUP[q]} for t in terms]
g04 = pd.concat([g04, pd.DataFrame(ru_rows)], ignore_index=True)

def chain(d):
    a = d[d.term == ANCHOR[d.group.iloc[0]]]['share']
    d = d.copy(); d['rel'] = d['share'] / a.iloc[0] if len(a) and a.iloc[0] > 0 else np.nan
    return d
rel = pd.concat([chain(d) for _, d in g04.groupby(['query_id', 'geoCode'])], ignore_index=True)
rel = rel.groupby(['group', 'term', 'geoCode'])['rel'].mean().unstack('geoCode').reindex(columns=GEOS)
profile = rel * 100          # якорь группы = 100; «250» = ищут в 2,5 раза чаще якоря
country_profile = profile.reset_index()
country_profile.round(2).to_csv(clean_path('country_profile.csv'), index=False)
print('country_profile.csv:', country_profile.shape, '| Россия есть в выгрузках:', sorted(set(r['query_id'] for r in ru_rows)))

In [ ]:
# Главный для слайда 7: heatmap «слово × страна» (доля внутри группы, %)
show_groups = [('ingredient', 'Ингредиенты'), ('category', 'Категории'), ('brand', 'Бренды')]
fig, axes = plt.subplots(1, 3, figsize=(14, 6.5), gridspec_kw={'width_ratios': [1, 1, 1]})
for ax, (grp, title) in zip(axes, show_groups):
    P = profile.loc[grp].copy()
    P = P.loc[P.mean(axis=1).sort_values(ascending=False).index].head(12)
    im = ax.imshow(np.log10(P.values.clip(1)), cmap='YlOrRd', aspect='auto', vmin=0, vmax=np.log10(400))
    ax.set_xticks(range(len(GEOS)), [RU_NAME[g] for g in GEOS], rotation=30, ha='right')
    ax.set_yticks(range(len(P)), P.index)
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            v = P.iloc[i, j]
            ax.text(j, i, '—' if np.isnan(v) else f'{v:.0f}', ha='center', va='center', fontsize=8)
    ax.set_title(title, loc='left', fontsize=11)
fig.suptitle('Профиль спроса: интерес к слову относительно якоря = 100 (ниацинамид / сыворотка для лица / anua); последние 12 мес;\n'
             'Россия — Вордстат, остальные — Google Trends; «—» — нет данных',
             x=0.01, ha='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG / 'q3_heatmap_profile.png', bbox_inches='tight'); plt.show()

In [ ]:
# Какие страны похожи
# Косинусное сходство центрированных log-профилей (= корреляция Пирсона); попарно по словам, где есть обе страны
P = np.log1p(profile)
cos = P.corr(min_periods=8).rename(index=RU_NAME, columns=RU_NAME)
n_pairs = P.notna().astype(int).T @ P.notna().astype(int)
print('Слов в паре (мин–макс):', int(n_pairs.values.min()), '–', int(n_pairs.values.max())); cos.round(2)

In [ ]:
# Horizontal bar: тренд запросов России за последние 24 мес (% в год), значимые выделены
r24 = ru.dropna(subset=['trend24_year_pct']).copy()
r24 = r24[r24.group.isin(['ingredient', 'category'])].sort_values('trend24_year_pct')
r24 = pd.concat([r24.head(8), r24.tail(10)]).drop_duplicates('term')
colors = np.where(r24.trend24_p < 0.05, np.where(r24.trend24_year_pct > 0, '#D55E00', '#0072B2'), '#BBBBBB')
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(r24.term + np.where(r24.group == 'category', ' (кат.)', ''), r24.trend24_year_pct.clip(upper=400), color=colors)
ax.axvline(0, color='k', lw=0.6); ax.set_xlim(-75, 360)
for y, v in enumerate(r24.trend24_year_pct):
    ax.text(min(v, 400) + (5 if v >= 0 else -5), y, f'{v:+.0f}%', va='center', ha='left' if v >= 0 else 'right', fontsize=8)
ax.set_xlabel('тренд за 2024-11 … 2026-10, % в год (с поправкой на сезон)')
ax.set_title('Россия (Вордстат): что растёт и что падает\nцвет — значимо (p < 0,05, Newey–West), серый — нет', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG / 'q3_bar_russia_growth.png'); plt.show()

In [ ]:
# Small multiples: 5 общих ингредиентов по странам; каждый ряд — скользящее среднее за 3 прошлых мес, делённое на своё среднее
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2), sharey=True)
for ax, term in zip(axes, COMMON):
    for g in GEOS:
        d = (ws[ws.term == term] if g == 'RU' else ing[(ing.geo == g) & (ing.term == term)]).sort_values('month')
        if len(d) == 0:
            continue
        s = d['value'].rolling(3).mean(); s = s / s.mean()
        weak = d.zero_share.iloc[0] > 0.5
        ax.plot(pd.to_datetime(d.month), s, color=PALETTE[g], lw=2.2 if g == 'RU' else 1.2,
                ls=':' if weak else '-', alpha=0.5 if weak else 1, label=RU_NAME[g])
    ax.set_title(term, fontsize=10); ax.tick_params(axis='x', labelsize=8)
axes[0].set_ylabel('к среднему ряда (1 = обычно)')
axes[-1].legend(fontsize=8, frameon=False)
fig.suptitle('5 общих ингредиентов: направление по странам (пунктир — нулей > 50%, мало данных)', x=0.01, ha='left', fontsize=10)
fig.tight_layout(); fig.savefig(FIG / 'q3_small_multiples.png'); plt.show()

**Дополнительно: Корея как опережающий сигнал.** Naver DataLab показывает, что ищут в самой Корее. Если корейский интерес к ингредиенту опережает российский, то по Корее можно заранее видеть, что придёт в СНГ. Корреляция роста г/г: Корея t−k → Россия t, k = 0…12.

In [ ]:
nv = pd.read_csv(raw_path('naver_datalab_korea_monthly_2016-01_2026-10.csv'), encoding='utf-8-sig')
nv['term'] = nv.term.str.lower().replace({'витамин c': 'витамин с'})
rows = []
for term in sorted(set(nv.term) & set(ws.term)):
    kr = nv[nv.term == term].set_index('month')['value']
    rr = ws[ws.term == term].set_index('month')['value']
    kr_y, ru_y = yoy_log(kr), yoy_log(rr)
    kr_y.index, ru_y.index = kr.index, rr.index
    best_k, best_r = None, -1
    for k in range(0, 13):
        x = kr_y.shift(k).reindex(ru_y.index)
        m = x.notna() & ru_y.notna()
        if m.sum() > 24:
            r = x[m].corr(ru_y[m])
            if r > best_r: best_k, best_r = k, r
    rows.append({'ингредиент': term, 'лучший лаг Корея → Россия, мес': best_k, 'r': best_r})
korea_lead = pd.DataFrame(rows).sort_values('r', ascending=False).set_index('ингредиент')
korea_lead.round(2).head(10)

In [ ]:
# Сводка для вывода (числа подставляются из таблиц выше)
top_ru = ru[(ru.trend24_p < 0.05) & (ru.trend24_year_pct > 0)].sort_values('trend24_year_pct', ascending=False)
down_ru = ru[(ru.trend24_p < 0.05) & (ru.trend24_year_pct < 0)].sort_values('trend24_year_pct')
print('Россия, значимо растут (24 мес):', ', '.join(f"{t} {v:+.0f}%" for t, v in zip(top_ru.term, top_ru.trend24_year_pct)))
print('Россия, значимо падают (24 мес):', ', '.join(f"{t} {v:+.0f}%" for t, v in zip(down_ru.term, down_ru.trend24_year_pct)))
for g in ['KZ', 'BY', 'KG', 'UZ']:
    o = other[(other.geo == g) & other.trend24_p.notna()].sort_values('trend24_year_pct', ascending=False)
    print(RU_NAME[g] + ':', ', '.join(f"{t} {v:+.0f}%{'*' if p < .05 else ''}" for t, v, p in zip(o.term, o.trend24_year_pct, o.trend24_p)))
print('\nСамые похожие профили:', cos.where(~np.eye(len(cos), dtype=bool)).stack().idxmax(),
      round(cos.where(~np.eye(len(cos), dtype=bool)).stack().max(), 2))

### Вывод по вопросу 3 (для стратегии)

Числа — на данных 26.11.2026; тренд за последние 24 мес с поправкой на сезон, * — значимо (Newey–West, p < 0,05).

- **Россия, растут:** PDRN (+309% в год), транексамовая кислота (+99%), спикулы (+71%), бакучиол и азелаиновая кислота (+34%), центелла (+24%), пептиды (+23%); категории — пэды (+44%) и тонеры (+23%); бренды — Medicube (+505%), Numbuzin (+161%), Anua (+102%), Skin1004 (+76%). Доля ингредиентов в запросах выросла с 11% (2020) до 26% (2026): покупатель выбирает по составу.
- **Россия, падают:** прополис (−30%), чайное дерево (−26%), гидрофильное масло (−22%), муцин улитки (−18%), кушоны (−15%), тканевые маски (−13%); бренды Manyo, Dr.Jart, Missha. Это «старые хиты» K-beauty.
- **Казахстан:** растут азелаиновая кислота (+64%*), центелла (+16%*), ретинол (+6%*), ниацинамид падает (−10%*). **Беларусь:** азелаиновая (+26%*), ретинол (+14%*). Кыргызстан и Узбекистан — мало сигнала, значим только ретинол в Узбекистане (+12%*).
- **Профили стран** (снимок 12 мес): Казахстан, Беларусь, Кыргызстан и Узбекистан похожи между собой (сходство 0,80–0,93; Казахстан–Беларусь 0,93) и заметно отличаются от России (0,62–0,74). В Центральной Азии относительно ниацинамида ищут больше ретинола (в 2,3–3,3 раза), пептидов и азелаиновой кислоты, в категориях лидирует крем для лица; в России — ниацинамид и PDRN (ретинола в 2 раза меньше ниацинамида), бренд №1 — Medicube (в Казахстане — Anua и Round Lab).
- **Корея опережает Россию на 8–12 месяцев** по PDRN (r = 0,68), спикулам (0,60), центелле (0,41), коллагену (0,41), ретинолу (0,39): что растёт в Корее сейчас, — кандидат для СНГ через год. Сверяем со списком «вырастет» Леры (вопрос 4).
- **Линейка:** Россия — PDRN, спикулы, транексамовая и азелаиновая кислоты, пэды и тонеры (бренды-ориентиры Medicube, Anua); Казахстан + Беларусь (+ Кыргызстан, Узбекистан) — **одна линейка**: ретинол, пептиды, азелаиновая кислота, центелла, крем для лица.

**Ограничения:** запросы по ингредиентам и категориям — спрос на косметику вообще, не только корейскую; категории по месяцам есть только для России; Google Trends — относительный индекс, сравним только внутри своей выгрузки; у Беларуси нет данных по брендам, у Кыргызстана — по PDRN, ретинолу, центелле в снимке.